In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from dotenv import load_dotenv
import os

In [9]:
load_dotenv("../.env")
census_key = os.getenv("CENSUS_API_KEY")

In [10]:
API_KEY    = census_key
STATE_FIPS = "54"                          # West Virginia
YEARS      = [2019, 2020, 2021, 2022, 2023]

# S1201_C02_001E = Percent never married
# S1201_C05_001E = Percent divorced
VARIABLES  = "S1201_C02_001E,S1201_C05_001E"

In [11]:
# ── STEP 1: Pull ACS 5-Year S1201 for all WV counties ───────────────────────
records = []

for year in YEARS:
    url = (
        f"https://api.census.gov/data/{year}/acs/acs5/subject"
        f"?get=NAME,{VARIABLES}"
        f"&for=county:*"
        f"&in=state:{STATE_FIPS}"
        f"&key={API_KEY}"
    )
    response = requests.get(url)

    # Debug print in case of errors
    print(f"Year: {year} | Status: {response.status_code}")
    if response.status_code != 200:
        print(f"Response: {response.text[:300]}")
        continue

    data = response.json()
    headers = data[0]
    for row in data[1:]:
        record = dict(zip(headers, row))
        record["Year"] = year
        records.append(record)

df = pd.DataFrame(records)
df

Year: 2019 | Status: 200
Year: 2020 | Status: 200
Year: 2021 | Status: 200
Year: 2022 | Status: 200
Year: 2023 | Status: 200


,NAME,S1201_C02_001E,S1201_C05_001E,state,county,Year
0,"Summers County, West Virginia",53.7,1.1,54,089,2019
1,"Greenbrier County, West Virginia",47.8,1.2,54,025,2019
2,"Mineral County, West Virginia",52.5,1.5,54,057,2019
3,"Lewis County, West Virginia",53.2,1.7,54,041,2019
4,"Pocahontas County, West Virginia",50.5,1.6,54,075,2019
...,...,...,...,...,...,...
270,"Webster County, West Virginia",54.3,1.2,54,101,2023
271,"Wetzel County, West Virginia",49.1,2.0,54,103,2023
272,"Wirt County, West Virginia",57.8,0.1,54,105,2023
273,"Wood County, West Virginia",48.6,1.1,54,107,2023


In [12]:
df["FIPS_Code"]        = df["state"] + df["county"]
df["County"]           = df["NAME"].str.replace(", West Virginia", "", regex=False)
df["Pct_Never_Married"] = pd.to_numeric(df["S1201_C02_001E"], errors="coerce")
df["Pct_Divorced"]      = pd.to_numeric(df["S1201_C05_001E"], errors="coerce")

# ── STEP 3: Final structure ───────────────────────────────────────────────────
df = df[["Year", "FIPS_Code", "County", "Pct_Never_Married", "Pct_Divorced"]].copy()
df = df.sort_values(["Year", "FIPS_Code"]).reset_index(drop=True)

df

,Year,FIPS_Code,County,Pct_Never_Married,Pct_Divorced
0,2019,54001,Barbour County,48.3,1.3
1,2019,54003,Berkeley County,51.9,1.9
2,2019,54005,Boone County,53.4,1.1
3,2019,54007,Braxton County,54.6,1.9
4,2019,54009,Brooke County,49.7,1.5
...,...,...,...,...,...
270,2023,54101,Webster County,54.3,1.2
271,2023,54103,Wetzel County,49.1,2.0
272,2023,54105,Wirt County,57.8,0.1
273,2023,54107,Wood County,48.6,1.1


In [13]:
df.to_csv("marital_status.csv", index=False)